# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant JSON-LD URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Title: {metadata.name}\nDescription: {metadata.description}\n")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets and their fields by `@id`
record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record sets:\n")
for rs in record_sets:
    print(f"➡️ RecordSet: {rs['@id']}")
    # List fields
    fields = rs.get('field', [])
    # Normalize to list
    if isinstance(fields, dict):
        fields = [fields]
    print("  Fields (by @id):")
    for field in fields:
        field_id = field.get('@id') if isinstance(field, dict) else field
        print(f"    - {field_id}")
    print('')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Let's choose the main protein abundance record set.
# We'll select the RecordSet with the most fields or with 'protein' or 'abundance' in its name/ID for demonstration.
record_set_ids = [rs['@id'] for rs in record_sets]
# For demonstration, simply use the first found RecordSet
if record_set_ids:
    chosen_record_set = record_set_ids[0]
else:
    raise RuntimeError("No record sets found in the metadata.")

# Extract data from all record sets into a dictionary of DataFrames
dataframes = {}
for rs in record_set_ids:
    print(f"Loading records from RecordSet: {rs}")
    records = list(dataset.records(record_set=rs))
    if records:
        dataframes[rs] = pd.DataFrame(records)
    else:
        dataframes[rs] = pd.DataFrame()
    print(f" - {len(records)} records loaded.")
    if not dataframes[rs].empty:
        print(f" - Columns: {dataframes[rs].columns.tolist()}")
    print()
# For EDA, we'll continue with the chosen_record_set if it contains data

if not dataframes[chosen_record_set].empty:
    print(f"Columns in {chosen_record_set}:")
    print(dataframes[chosen_record_set].columns.tolist())
    display(dataframes[chosen_record_set].head())
else:
    print(f"No data found in record set {chosen_record_set}.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Attempt common analyses on numeric fields if they exist.
import numpy as np

df = dataframes[chosen_record_set]
print(f">> Exploring record set: {chosen_record_set}\n")

# Detect numeric columns and select one
numeric_fields = df.select_dtypes(include=np.number).columns.tolist()
if not numeric_fields:
    # Try to convert anything that looks numeric
    print("No numeric columns detected. Attempting to convert numeric-looking columns...")
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='ignore')
    numeric_fields = df.select_dtypes(include=np.number).columns.tolist()

if numeric_fields:
    numeric_field = numeric_fields[0]  # select the first
    print(f"Using numeric field for EDA: {numeric_field}")
    threshold = df[numeric_field].mean()  # set a threshold to the mean for demonstration
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())

    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, norm_col]].head())

    # Try grouping by the next categorical or string field
    group_field = None
    for col in df.columns:
        if col != numeric_field and df[col].dtype == object:
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped data by {group_field} (showing means of {numeric_field}):")
        display(grouped_df.head())
    else:
        print("No suitable categorical field found for grouping.")
else:
    print("No numeric fields found for analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Only run visualization if we had a numeric field
if numeric_fields and not df.empty:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()
    
    if group_field:
        # Boxplot by group_field for normalized field
        plt.figure(figsize=(10,6))
        sns.boxplot(data=filtered_df, x=group_field, y=norm_col)
        plt.title(f"{numeric_field} (normalized) by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We loaded the FAIR^2 dataset "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells" via its Croissant schema using `mlcroissant`.
- The dataset contains one or more structured record sets as defined by the Croissant schema, with tabular columns representing protein abundances and related features.
- Example explorations included filtering by numeric fields (e.g., abundance or peptide counts), normalizing and visualizing distributions, and grouping by categorical properties.
- For more advanced analysis, further exploration of the full Croissant metadata (record set and field `@id`s), and tailoring EDA to the specific biological context, is encouraged.

#### References
- [Croissant Standard Documentation](https://mlcommons.org/croissant/)
- [mlcroissant Python library](https://github.com/mlcommons/croissant)
- [Dataset FAIR metadata](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)